# RQ4: Unsafe-to-Safe Conversion Dataset

This notebook experiments with extracting a training and evaluation dataset for an AI system that detects Rust `unsafe` code blocks that can be converted to safe code and then proposes the conversion.

The core idea is to build block-level examples from the collected vulnerability tables, pair vulnerable unsafe blocks with their fixed-version counterparts, enrich the rows with commit and category context, and then derive useful benchmark views for model training and evaluation.

All relational data manipulation (joins, aggregations, candidate generation) is done in SQL against the SQLite database; pandas is used only for display and for the git/filesystem-based source extraction.

**Core concept:** a vulnerable unsafe block is a *true unsafe-to-safe conversion* when its containing function no longer has any unsafe blocks in the fixed version. This is computed directly in SQL from the `unsafe_block` / `unsafe_block_fix` tables — no git/filesystem needed for the label.

In [12]:
import ast
import json
import random
import re
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append('../utils')
import database as db
from utils import get_full_project_name

pd.set_option('display.max_colwidth', 120)

repo_root = Path('..').resolve()
repos_mirror = repo_root / 'repos_mirror'
repos_worktree = repo_root / 'repos_worktree'
data_file = repo_root / 'data_extraction' / 'fix_commits2.csv'

span_pattern = re.compile(r'^(?P<path>.+?):\s*(?P<start>\d+)-(?P<end>\d+)$')


def parse_span(span_text):
    match = span_pattern.match(str(span_text).strip())
    if match is None:
        return None, None, None
    return match.group('path'), int(match.group('start')), int(match.group('end'))


def safe_file_count(repo_path):
    if not repo_path.exists():
        return 0, []
    rust_files = []
    for path in repo_path.rglob('*.rs'):
        path_text = path.as_posix()
        if '/target/' in path_text or '/.git/' in path_text:
            continue
        rust_files.append(path)
    return len(rust_files), rust_files[:20]


def find_unsafe_files(repo_path, limit=200):
    candidates = []
    if not repo_path.exists():
        return candidates
    for path in repo_path.rglob('*.rs'):
        path_text = path.as_posix()
        if '/target/' in path_text or '/.git/' in path_text:
            continue
        try:
            text = path.read_text(encoding='utf-8', errors='replace')
        except Exception:
            continue
        if re.search(r'\bunsafe\b|\*const\b|\*mut\b', text):
            candidates.append(path)
        if len(candidates) >= limit:
            break
    return candidates

## 1. Repository Ingestion and File Discovery

Get row counts directly from the database via SQL, load the commit list, then identify Rust files that are likely to contain unsafe code or raw-pointer usage.

In [13]:
def sql_count(table):
    return pd.read_sql(f'SELECT COUNT(*) AS n FROM {table}', con=db.conn)['n'].iloc[0]

print('commits:', sql_count('commits'))
print('unsafe blocks:', sql_count('unsafe_block'))
print('fixed unsafe blocks:', sql_count('unsafe_block_fix'))
print('functions:', sql_count('function'))
print('fixed functions:', sql_count('function_fix'))
print('file changes:', sql_count('file_change'))
print('types:', sql_count('cve_sfp'))

df_commits = pd.read_sql('''
    SELECT c.cve_id, c.hash, c.repo_url, cv.package, cv.severity
    FROM commits AS c
    JOIN cve AS cv ON c.cve_id = cv.id
''', con=db.conn)

# Removed unsafe blocks: vulnerable unsafe blocks whose (hash, path, fn_id) has
# no counterpart in the fixed version (unsafe_block_fix). This is the block-level
# view of unsafe-to-safe conversion - the block itself was removed by the fix,
# even if its containing function still holds other unsafe blocks.
REMOVED_BLOCKS_SQL = """
SELECT unfixed.cve_id, unfixed.hash, unfixed.path, unfixed.fn_id,
       unfixed.span_start, unfixed.span_end, c.repo_url
FROM unsafe_block AS unfixed
LEFT JOIN unsafe_block_fix AS fixed
    ON unfixed.hash = fixed.hash
   AND unfixed.path = fixed.path
   AND unfixed.fn_id = fixed.fn_id
LEFT JOIN commits c ON unfixed.hash = c.hash
WHERE fixed.hash IS NULL
"""
df_removed = pd.read_sql(REMOVED_BLOCKS_SQL, con=db.conn)
print('removed unsafe blocks (block-level):', len(df_removed))

sample_repos = [get_full_project_name(url) for url in df_commits['repo_url'].dropna().drop_duplicates().head(5)]
for repo_name in sample_repos:
    repo_path = repos_mirror / repo_name
    rust_count, rust_examples = safe_file_count(repo_path)
    unsafe_candidates = find_unsafe_files(repo_path, limit=20)
    print('\nrepo:', repo_name)
    print('  rust files:', rust_count)
    print('  unsafe-like files:', len(unsafe_candidates))
    print('  sample rust files:', [p.as_posix().replace(str(repo_path) + '/', '') for p in rust_examples[:5]])
    print('  sample unsafe files:', [p.as_posix().replace(str(repo_path) + '/', '') for p in unsafe_candidates[:5]])

commits: 742
unsafe blocks: 64254
fixed unsafe blocks: 64380
functions: 920601
fixed functions: 921212
file changes: 3255
types: 1790
removed unsafe blocks (block-level): 114

repo: containers_aardvark-dns
  rust files: 0
  unsafe-like files: 0
  sample rust files: []
  sample unsafe files: []

repo: rodrimati1992_abi_stable_crates
  rust files: 0
  unsafe-like files: 0
  sample rust files: []
  sample unsafe files: []

repo: LemmyNet_activitypub-federation-rust
  rust files: 0
  unsafe-like files: 0
  sample rust files: []
  sample unsafe files: []

repo: actix_actix-net
  rust files: 0
  unsafe-like files: 0
  sample rust files: []
  sample unsafe files: []

repo: actix_actix-web
  rust files: 0
  unsafe-like files: 0
  sample rust files: []
  sample unsafe files: []


## 2. Unsafe Block Extraction and Conversion Label

Build the block-level candidate manifest in a single SQL query. The conversion label is computed from the ground up:

- `converted_to_safe = 1` when the vulnerable block's **containing function** (`cve_id, hash, fn_id, path`) has **no** unsafe blocks in the fixed version (`unsafe_block_fix`).
- `converted_to_safe = 0` when the function still has unsafe blocks after the fix.

The query also joins commit metadata, vulnerability types, file-change context, and per-file function summaries.

In [ ]:
CANDIDATES_SQL = """
WITH
vul AS (
    SELECT ub.cve_id, ub.hash, ub.fn_id, ub.path, ub.span_start, ub.span_end, ub.unsafety,
           c.repo_url, s.sfp_category
    FROM unsafe_block ub
    LEFT JOIN commits c ON ub.hash = c.hash
    LEFT JOIN cve_sfp s ON ub.cve_id = s.cve_id
),
fn_fix_unsafe AS (
    -- Functions that still contain unsafe blocks after the fix.
    SELECT DISTINCT cve_id, hash, fn_id, path
    FROM unsafe_block_fix
),
block_gone AS (
    -- Block-level anti-join (your query): each vulnerable unsafe block whose
    -- (hash, path, fn_id) has NO counterpart in the fixed version. Exact block
    -- identity includes its span, so still-present sibling blocks stay excluded.
    SELECT ub.cve_id, ub.hash, ub.path, ub.fn_id, ub.span_start, ub.span_end
    FROM unsafe_block ub
    LEFT JOIN unsafe_block_fix fixed
        ON ub.hash = fixed.hash AND ub.path = fixed.path AND ub.fn_id = fixed.fn_id
    WHERE fixed.hash IS NULL
),
chg AS (
    SELECT hash, path,
           GROUP_CONCAT(DISTINCT change_type) AS change_type,
           MAX(CASE WHEN change_type LIKE '%RENAME%' THEN 1 ELSE 0 END) AS was_renamed,
           MAX(CAST(num_lines_added   AS INTEGER)) AS num_lines_added,
           MAX(CAST(num_lines_deleted AS INTEGER)) AS num_lines_deleted,
           MAX(CAST(nloc              AS INTEGER)) AS nloc
    FROM (
        SELECT hash, old_path AS path, change_type, num_lines_added, num_lines_deleted, nloc
        FROM file_change WHERE old_path IS NOT NULL
        UNION ALL
        SELECT hash, new_path AS path, change_type, num_lines_added, num_lines_deleted, nloc
        FROM file_change WHERE new_path IS NOT NULL
    )
    GROUP BY hash, path
),
fn_fix_safe AS (
    -- Functions that are safe (unsafety='False') in the fixed version.
    SELECT DISTINCT cve_id, hash, path, name
    FROM function_fix WHERE unsafety = 'False'
),
fn_vul AS (
    SELECT cve_id, hash, path,
           SUM(CASE WHEN unsafety = 'False' THEN 1 ELSE 0 END) AS safe_funcs_in_file,
           SUM(CASE WHEN unsafety = 'True'  THEN 1 ELSE 0 END) AS unsafe_funcs_in_file,
           COUNT(*) AS total_funcs_in_file
    FROM function GROUP BY cve_id, hash, path
),
fn_fix AS (
    SELECT cve_id, hash, path,
           SUM(CASE WHEN unsafety = 'False' THEN 1 ELSE 0 END) AS safe_funcs_fix_in_file,
           SUM(CASE WHEN unsafety = 'True'  THEN 1 ELSE 0 END) AS unsafe_funcs_fix_in_file,
           COUNT(*) AS total_funcs_fix_in_file
    FROM function_fix GROUP BY cve_id, hash, path
)
SELECT vul.cve_id, vul.hash, vul.fn_id, vul.path, vul.span_start, vul.span_end, vul.unsafety,
       vul.repo_url, vul.sfp_category,
       CASE WHEN fn_fix_unsafe.fn_id IS NULL THEN 1 ELSE 0 END AS converted_to_safe,
       CASE WHEN fn_fix_unsafe.fn_id IS NULL THEN 'converted_to_safe' ELSE 'still_unsafe' END AS conversion_label,
       -- DB-based verification: exclude renamed files (rename artifact, not a real conversion)
       -- and require the function to be safe in the fixed version.
       CASE WHEN fn_fix_unsafe.fn_id IS NULL
                 AND COALESCE(chg.was_renamed, 0) = 0
                 AND fn_fix_safe.name IS NOT NULL
            THEN 1 ELSE 0 END AS verified_conversion,
       -- Block-level label: 1 when this block itself disappeared after the fix.
       CASE WHEN block_gone.fn_id IS NOT NULL THEN 1 ELSE 0 END AS block_disappeared,
       chg.change_type,
       COALESCE(chg.was_renamed, 0) AS was_renamed,
       COALESCE(chg.num_lines_added, 0)   AS num_lines_added,
       COALESCE(chg.num_lines_deleted, 0) AS num_lines_deleted,
       COALESCE(chg.nloc, 0)              AS nloc,
       COALESCE(fn_vul.safe_funcs_in_file, 0)   AS safe_funcs_in_file,
       COALESCE(fn_vul.unsafe_funcs_in_file, 0) AS unsafe_funcs_in_file,
       COALESCE(fn_vul.total_funcs_in_file, 0)  AS total_funcs_in_file,
       COALESCE(fn_fix.safe_funcs_fix_in_file, 0)   AS safe_funcs_fix_in_file,
       COALESCE(fn_fix.unsafe_funcs_fix_in_file, 0) AS unsafe_funcs_fix_in_file,
       COALESCE(fn_fix.total_funcs_fix_in_file, 0)  AS total_funcs_fix_in_file
FROM vul
LEFT JOIN fn_fix_unsafe ON vul.cve_id = fn_fix_unsafe.cve_id
                       AND vul.hash = fn_fix_unsafe.hash
                       AND vul.fn_id = fn_fix_unsafe.fn_id
                       AND vul.path = fn_fix_unsafe.path
LEFT JOIN fn_fix_safe ON vul.cve_id = fn_fix_safe.cve_id
                     AND vul.hash = fn_fix_safe.hash
                     AND vul.path = fn_fix_safe.path
                     AND fn_fix_safe.name = substr(vul.fn_id, instr(vul.fn_id, '::') + 2)
LEFT JOIN block_gone ON vul.cve_id = block_gone.cve_id
                     AND vul.hash = block_gone.hash
                     AND vul.path = block_gone.path
                     AND vul.fn_id = block_gone.fn_id
                     AND vul.span_start = block_gone.span_start
                     AND vul.span_end = block_gone.span_end
LEFT JOIN chg    ON vul.hash = chg.hash AND vul.path = chg.path
LEFT JOIN fn_vul ON vul.cve_id = fn_vul.cve_id AND vul.hash = fn_vul.hash AND vul.path = fn_vul.path
LEFT JOIN fn_fix ON vul.cve_id = fn_fix.cve_id AND vul.hash = fn_fix.hash AND vul.path = fn_fix.path
"""

df_candidates = pd.read_sql(CANDIDATES_SQL, con=db.conn)

# Load per-file diffs already parsed by pydriller: flat {'added': [(line, text)],
# 'deleted': [(line, text)]} lists, with comment/blank lines pre-stripped by
# extract_changes.eliminate_comment_diff. No raw-diff re-parsing needed.
df_diffs = pd.read_sql('''
    SELECT hash, old_path, new_path, diff_parsed
    FROM file_change
''', con=db.conn)

def parse_diff_parsed(text):
    """Parse the DB's diff_parsed TEXT into {'added': [(line, text)], 'deleted': [(line, text)]}."""
    if not isinstance(text, str) or not text.strip():
        return None
    try:
        return ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return None

# Build a lookup: (hash, path) -> list of parsed diff dicts.
diff_lookup = {}
for _, r in df_diffs.iterrows():
    parsed = parse_diff_parsed(r['diff_parsed'])
    if parsed is None:
        continue
    for path in [r['old_path'], r['new_path']]:
        if pd.notna(path):
            diff_lookup.setdefault((r['hash'], path), []).append(parsed)

def align_block_to_diff(block_start, block_end, parsed_list, window=5):
    """Return (deleted_lines, added_lines) for the block's span, using diff_parsed.
    unsafe_snippet = deleted lines with line numbers inside the block span;
    safe_snippet = added lines within +-window lines of the block span.
    ponytail: window pairing is approximate -- diff_parsed is flattened (no hunk
    boundaries), so added lines are not guaranteed to be the exact replacements.
    Upgrade path: re-derive hunks from file_change.diff when exact same-hunk pairs
    are required.
    Returns None if no deleted lines fall within the block span."""
    del_lines = []
    for parsed in parsed_list:
        for n, text in parsed.get('deleted', []):
            if block_start <= n <= block_end:
                del_lines.append(text)
    if not del_lines:
        return None
    add_lines = []
    for parsed in parsed_list:
        for n, text in parsed.get('added', []):
            if block_start - window <= n <= block_end + window:
                add_lines.append(text)
    return del_lines, add_lines

def add_source_context(frame, limit=200):
    """Build aligned before/after snippets from diff_parsed (DB): unsafe = deleted lines
    in the block span, safe = added lines within +-5 lines of the block span."""
    rows = []
    for _, row in frame.head(limit).iterrows():
        diff = diff_lookup.get((row['hash'], row['path']))
        if diff is None:
            continue
        aligned = align_block_to_diff(int(row['span_start']), int(row['span_end']), diff)
        if aligned is None:
            continue  # no deleted lines in this block's span -> no real before/after pair
        deleted_lines, added_lines = aligned
        rows.append({
            'cve_id': row['cve_id'],
            'hash': row['hash'],
            'repo_name': get_full_project_name(row['repo_url']) if pd.notna(row.get('repo_url')) else None,
            'path': row['path'],
            'sfp_category': row.get('sfp_category'),
            'unsafe_block_source': '\n'.join(deleted_lines),
            'fixed_source': '\n'.join(added_lines),
        })
    return pd.DataFrame(rows)

# Sample from verified conversions (which have diff-aligned pairs), not all candidates.
verified_mask = df_candidates['verified_conversion'] == 1
sample_context = df_candidates[verified_mask].sample(
    min(50, int(verified_mask.sum())), random_state=42
).reset_index(drop=True)
df_snippets = add_source_context(sample_context, limit=50)

print('sampled unsafe blocks:', len(sample_context))
print('snippet rows extracted (aligned via diff_parsed):', len(df_snippets))
print(df_snippets[['cve_id', 'repo_name', 'path', 'sfp_category']].head(10).to_string(index=False))

sampled unsafe blocks: 50
snippet rows extracted (aligned via diff_parsed): 37
cve_id            repo_name                                                                 path         sfp_category
    24      actix_actix-net                                              actix-utils/src/mpsc.rs  Resource Management
  1131     succinctlabs_sp1   crates/recursion/core/src/poseidon2_wide/columns/syscall_params.rs                  NaN
   531 andrewhickman_id-map                                                           src/lib.rs Exception Management
  1131     succinctlabs_sp1 crates/recursion/core/src/poseidon2_wide/columns/opcode_workspace.rs                  NaN
    24      actix_actix-net                                              actix-utils/src/cell.rs  Resource Management
   160         Enet4_bra-rs                                                        src/greedy.rs        Memory Access
   531 andrewhickman_id-map                                                           src/lib.r

## 2.5 Block-Level Conversion Candidates

The block-level anti-join (`REMOVED_BLOCKS_SQL`) finds vulnerable unsafe blocks whose `(hash, path, fn_id)` has no counterpart in the fixed version -- the block itself was removed by the fix, regardless of whether its containing function still holds other unsafe blocks.

This is a stricter, complementary view to the function-level `converted_to_safe` label, and directly feeds the pair construction below.

In [15]:
# Overlap between block-level disappearance and function-level conversion.
# Per-function flags are deduplicated on the function key (no cve_sfp fan-out),
# so the counts below stay at block granularity: 114 disappeared blocks total.
function_flags = df_candidates[[
    'cve_id', 'hash', 'path', 'fn_id', 'converted_to_safe', 'verified_conversion'
]].drop_duplicates()
df_removed = df_removed.merge(function_flags, on=['cve_id', 'hash', 'path', 'fn_id'], how='left')
df_removed['is_function_level_conversion'] = df_removed['converted_to_safe'].fillna(0).astype(int)
df_removed['block_disappeared'] = 1

print('block-level disappeared rows:', len(df_removed))
print('  also function-level converted_to_safe:', int(df_removed['is_function_level_conversion'].sum()))
print('  block-level only (function still has unsafe blocks after fix):',
      int((df_removed['is_function_level_conversion'] == 0).sum()))
# By type: one row per block per category (a multi-category CVE counts the block once per category).
by_type = (df_candidates[df_candidates['block_disappeared'] == 1]
           .drop_duplicates(['cve_id', 'hash', 'fn_id', 'path', 'span_start', 'span_end', 'sfp_category'])
           ['sfp_category'].value_counts(dropna=False))
print('\nby type:')
print(by_type.head(10).to_string())
print('\npreview:')
preview_cols = ['cve_id', 'hash', 'path', 'fn_id', 'span_start', 'span_end', 'is_function_level_conversion']
print(df_removed[preview_cols].head(10).to_string(index=False))

block-level disappeared rows: 114
  also function-level converted_to_safe: 114
  block-level only (function still has unsafe blocks after fix): 0

by type:
sfp_category
Exception Management    43
NaN                     37
Memory Management       12
Tainted Input            8
Memory Access            7
Synchronization          4
Resource Management      3

preview:
cve_id                                     hash                           path                  fn_id span_start span_end  is_function_level_conversion
    17 c41b5d8dd4235ccca84d0b687996615c0c64d956      actix-codec/src/framed.rs      framed::next_item        252      254                             1
    17 c41b5d8dd4235ccca84d0b687996615c0c64d956      actix-codec/src/framed.rs          framed::close        307      310                             1
    17 c41b5d8dd4235ccca84d0b687996615c0c64d956        actix-rt/src/arbiter.rs         arbiter::spawn        184      184                             1
    23 a67e38b4a07c92a3c

## 3. Conversion Candidate Summary

Summarize the block-level candidate manifest produced by the SQL query: conversion-label distribution and breakdown by vulnerability type.

In [23]:
print('candidate rows:', len(df_candidates))
print(df_candidates['conversion_label'].value_counts(dropna=False).to_string())
print('\nby type:')
print(df_candidates.groupby('sfp_category')['converted_to_safe'].agg(['count', 'sum']).sort_values('count', ascending=False).head(10).to_string())
print('\npreview:')
preview_cols = ['cve_id', 'hash', 'path', 'sfp_category', 'conversion_label', 'change_type', 'num_lines_added', 'num_lines_deleted']
print(df_candidates[preview_cols].head(10).to_string(index=False))

candidate rows: 74603
conversion_label
still_unsafe         74421
converted_to_safe      182

by type:
                      count  sum
sfp_category                    
Memory Access         12981   19
Resource Management    8213   17
Tainted Input          6366    8
Memory Management      4573   53
Exception Management   4553   43
Cryptography           3868    0
Access Control         3399    0
Synchronization        3340    4
Information Leak       2064    0
Other                  2064    0

preview:
cve_id                                     hash                      path        sfp_category  conversion_label             change_type  num_lines_added  num_lines_deleted
     9 aa109bbd6743abd7027e589cc4b871dd2dce7d50       src/server/serve.rs Resource Management      still_unsafe                     NaN                0                  0
     9 aa109bbd6743abd7027e589cc4b871dd2dce7d50       src/commands/run.rs Resource Management      still_unsafe                     NaN            

## 4. Pair Construction: Unsafe vs. Safe Code

Filter the converted candidates in SQL, then align each unsafe block with its best available safe equivalent, store input-output pairs, and record labels for convertibility and transformation type.

In [24]:
# Two sources of converted-candidate blocks:
#   1. function-level: DB-verified conversions (function unsafe-block-free after the fix AND not renamed).
#   2. block-level:    anti-join -- blocks whose (hash, path, fn_id) has no fixed counterpart
df_func_manifest = pd.read_sql(f"""
    SELECT * FROM ({CANDIDATES_SQL})
    WHERE verified_conversion = 1
""", con=db.conn)
df_func_manifest = df_func_manifest.copy()
df_func_manifest['source_label'] = 'function_level'

df_block_manifest = pd.read_sql(f"""
    SELECT * FROM ({CANDIDATES_SQL})
    WHERE block_disappeared = 1
""", con=db.conn)
df_block_manifest = df_block_manifest.copy()
# Block-level label is carried by block_disappeared=1; the function-level
# converted_to_safe / conversion_label columns are kept as-is.
df_block_manifest['source_label'] = 'block_level'

# Combine; prefer the function-level row when the same block appears in both sources.
df_pairs_manifest = pd.concat([df_func_manifest, df_block_manifest], ignore_index=True)
df_pairs_manifest = df_pairs_manifest.drop_duplicates(
    subset=['cve_id', 'hash', 'fn_id', 'path', 'span_start', 'span_end'], keep='first'
)

# Extract source text for a smaller sample so the notebook can show real input-output examples.
sample_size = min(100, len(df_pairs_manifest))
df_pairs_sample = df_pairs_manifest.sample(sample_size, random_state=42).copy() if sample_size else df_pairs_manifest.head(0).copy()
df_pair_text = []
for _, row in df_pairs_sample.iterrows():
    diff = diff_lookup.get((row['hash'], row['path']))
    if diff is None:
        continue
    aligned = align_block_to_diff(int(row['span_start']), int(row['span_end']), diff)
    if aligned is None:
        continue  # no aligned diff lines -> skip (no real pair)
    deleted_lines, added_lines = aligned
    df_pair_text.append({
        'cve_id': row['cve_id'],
        'hash': row['hash'],
        'repo_name': get_full_project_name(row['repo_url']) if pd.notna(row.get('repo_url')) else None,
        'path': row['path'],
        'sfp_category': row.get('sfp_category'),
        'converted_to_safe': int(row['converted_to_safe']),
        'verified_conversion': int(row['verified_conversion']),
        'block_disappeared': int(row['block_disappeared']),
        'source_label': row['source_label'],
        'unsafe_snippet': '\n'.join(deleted_lines),
        'safe_snippet': '\n'.join(added_lines),
    })

df_pair_text = pd.DataFrame(df_pair_text)

manifest_path = repo_root / 'rq4_block_conversion_manifest.csv'
pair_sample_path = repo_root / 'rq4_block_conversion_pairs_sample.csv'
df_pairs_manifest.to_csv(manifest_path, index=False)
df_pair_text.to_csv(pair_sample_path, index=False)

print('converted pairs:', len(df_pairs_manifest))
print('  function-level (includes block-level overlap):',
      int((df_pairs_manifest['source_label'] == 'function_level').sum()))
print('  block-level only:', int((df_pairs_manifest['source_label'] == 'block_level').sum()))
print('pair-text rows extracted:', len(df_pair_text))
print('manifest saved to:', manifest_path)
print('sample pairs saved to:', pair_sample_path)
if not df_pair_text.empty:
    print('\nexample pair:')
    example_cols = ['cve_id', 'repo_name', 'path', 'sfp_category', 'source_label']
    print(df_pair_text[example_cols].head(5).to_string(index=False))

converted pairs: 114
  function-level (includes block-level overlap): 112
  block-level only: 2
pair-text rows extracted: 80
manifest saved to: /mnt/intel20/Code/rust_ecosystem/rq4_block_conversion_manifest.csv
sample pairs saved to: /mnt/intel20/Code/rust_ecosystem/rq4_block_conversion_pairs_sample.csv

example pair:
cve_id            repo_name                                                     path         sfp_category   source_label
  1131     succinctlabs_sp1 crates/recursion/core/src/cpu/columns/opcode_specific.rs                  NaN function_level
    23      actix_actix-net                                actix-service/src/cell.rs    Memory Management function_level
   531 andrewhickman_id-map                                               src/lib.rs Exception Management function_level
   963          risc0_risc0                       risc0/zkvm/platform/src/syscall.rs        Tainted Input function_level
    40        alloy-rs_core                      crates/sol-type-parser/src